# Retriever Comparison — TimeSeriesExam

Experiment: `retriever_comparison_v1`  
Models: ChatTS-8B · Qwen3-VL-8B-Instruct · Qwen3-8B · random_baseline  
Retrievers: none (0-shot) · random · text · vision_ts · ts  
Shots: 0 · 1 · 2 · 3 | Seeds: 0 · 1 · 2021

Data source: `../results/retriever_comparison_v1__*.json` (746 per-sample records per run)

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RESULTS_DIR = Path("../results")

## 1  Load data

In [3]:
records = []
for fpath in sorted(RESULTS_DIR.glob("retriever_comparison_v1__*.json")):
    records.extend(json.loads(fpath.read_text()))

df = pd.DataFrame(records)

MODEL_LABELS = {
    "bytedance-research/ChatTS-8B": "ChatTS-8B",
    "Qwen/Qwen3-VL-8B-Instruct":   "Qwen3-VL-8B",
    "Qwen/Qwen3-8B-vllm":           "Qwen3-8B",
    "random_baseline":              "Random",
}
df["model"] = df["method"].map(MODEL_LABELS).fillna(df["method"])
df["correct"] = df["correct"].astype(int)

n_runs = df.groupby(["method", "retriever", "num_shots", "seed"]).ngroups
print(f"Loaded {len(df):,} records from {n_runs} runs")

Loaded 520,708 records from 698 runs


### Coverage matrix

In [4]:
cov = (
    df.groupby(["model", "retriever", "num_shots"])["seed"]
    .nunique()
    .reset_index(name="n_seeds")
)
cov_pivot = cov.pivot_table(
    index=["model", "retriever"], columns="num_shots", values="n_seeds", fill_value=0
)
print(cov_pivot.to_string())

num_shots                                                            0    1    2    3    5    8
model         retriever                                                                        
ChatTS-8B     delay_dino                                           0.0  3.0  3.0  3.0  3.0  3.0
              none                                                 3.0  0.0  0.0  0.0  0.0  0.0
              random                                               0.0  3.0  3.0  3.0  3.0  3.0
              rrf                                                  0.0  3.0  3.0  3.0  3.0  3.0
              rrf-text-delay_dino                                  0.0  3.0  3.0  3.0  3.0  3.0
              rrf-text-ts-vision_ts-spectral-stats-vision_wavelet  0.0  3.0  3.0  3.0  3.0  3.0
              rrf-text-vision_ts                                   0.0  3.0  3.0  3.0  3.0  3.0
              rrf-ts-delay_dino                                    0.0  3.0  3.0  3.0  3.0  3.0
              rrf-ts-vision_ts          

## 2  Aggregation (seed-averaged accuracy)

In [5]:
RETRIEVER_ORDER = ["none", "random", "text", "vision_ts", "ts", "rrf", "delay_dino" , "rrf-text-delay_dino" , "rrf-ts-delay_dino",
"rrf-text-ts-vision_ts-spectral-stats-vision_wavelet", "rrf-text-vision_ts" , "rrf-ts-vision_ts" , "rrf-ts-vision_ts-delay_dino" , 
"rrf-vision_ts-delay_dino", "text_bge"]
MODELS_ORDER    = ["ChatTS-8B", "Qwen3-VL-8B", "Qwen3-8B", "Random"]
DIFF_ORDER      = ["easy", "medium", "hard"]

run_acc = (
    df.groupby(["model", "retriever", "num_shots", "seed"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)

mean_acc = (
    run_acc.groupby(["model", "retriever", "num_shots"])["accuracy"]
    .agg(acc_mean="mean", acc_std="std", n_seeds="count")
    .reset_index()
)
mean_acc["acc_std"] = mean_acc["acc_std"].fillna(0)

baseline = (
    mean_acc[mean_acc["retriever"] == "none"][["model", "acc_mean"]]
    .rename(columns={"acc_mean": "acc_0shot"})
)
mean_acc = mean_acc.merge(baseline, on="model", how="left")
mean_acc["delta"] = mean_acc["acc_mean"] - mean_acc["acc_0shot"]

print(f"{len(mean_acc)} (model × retriever × shots) combinations")

235 (model × retriever × shots) combinations


---
## 3  Overall accuracy — one table per model

Rows: retriever · Columns: shots · Values: seed-averaged accuracy (±std when >1 seed)  
`none` = 0-shot baseline (only has a 0-shot column)

In [6]:
TABLE_STYLES = [
    {"selector": "th.col_heading", "props": [
        ("background-color", "#2e2e2e"), ("color", "white"),
        ("font-size", "13px"), ("text-align", "center"),
        ("padding", "8px 16px"), ("border", "1px solid #444"),
    ]},
    {"selector": "th.row_heading", "props": [
        ("background-color", "#4a4a4a"), ("color", "white"),
        ("text-align", "center"), ("padding", "7px 14px"),
        ("border", "1px solid #666"), ("font-weight", "normal"),
    ]},
    {"selector": "th.blank", "props": [
        ("background-color", "#2e2e2e"), ("border", "1px solid #444"),
    ]},
    {"selector": "td", "props": [
        ("border", "1px solid #ddd"), ("padding", "7px 14px"),
    ]},
    {"selector": "caption", "props": [
        ("font-size", "15px"), ("font-weight", "bold"),
        ("text-align", "left"), ("padding", "10px 2px 5px"),
        ("color", "currentColor"),       # inherits theme text colour
        ("caption-side", "top"),
    ]},
    {"selector": "", "props": [
        ("border-collapse", "collapse"), ("margin-bottom", "28px"),
    ]},
]


def _acc_color(v, lo=0.30, hi=0.70):
    if pd.isna(v):
        return "#efefef"
    t = max(0.0, min(1.0, (v - lo) / (hi - lo)))
    return f"rgb({int(240-t*90)},{int(240-t*40)},{int(240-t*130)})"


def _delta_color(v, bound=0.08):
    if pd.isna(v):
        return "#efefef"
    t = max(-1.0, min(1.0, v / bound))
    if t >= 0:
        return f"rgb({int(240-t*90)},{int(240-t*40)},{int(240-t*130)})"
    else:
        s = -t
        return f"rgb(240,{int(240-s*140)},{int(240-s*140)})"


def _fmt(mean, std):
    if pd.isna(mean):
        return ""
    s = f"{mean*100:.1f}%"
    if not pd.isna(std) and std > 0.001:
        s += f"  ±{std*100:.1f}"
    return s


def _fmt_delta(v, std=None):
    if pd.isna(v):
        return ""
    s = f"{v*100:+.1f}pp"
    if std is not None and not pd.isna(std) and std > 0.001:
        s += f"  ±{std*100:.1f}"
    return s


def build_model_table(model_df, retriever_order, title,
                      value_col="acc_mean", std_col="acc_std",
                      fmt_fn=None, color_fn=None):
    """
    One table for a single model.
    Rows = retrievers, Columns = 0-shot / 1-shot / 2-shot / 3-shot.
    """
    if fmt_fn is None:
        fmt_fn = _fmt
    if color_fn is None:
        color_fn = _acc_color

    shots = sorted(model_df["num_shots"].unique())
    retrievers_in = [r for r in retriever_order if r in model_df["retriever"].values]

    pivot_val = model_df.pivot_table(index="retriever", columns="num_shots", values=value_col)
    pivot_std = model_df.pivot_table(index="retriever", columns="num_shots", values=std_col) \
        if std_col in model_df.columns else None
    pivot_val = pivot_val.reindex(retrievers_in)
    if pivot_std is not None:
        pivot_std = pivot_std.reindex(retrievers_in)

    num_tbl = pivot_val.rename(columns={s: f"{s}-shot" for s in shots})

    str_data = {
        f"{s}-shot": [
            fmt_fn(
                pivot_val.at[r, s] if s in pivot_val.columns else np.nan,
                pivot_std.at[r, s] if (pivot_std is not None and s in pivot_std.columns) else np.nan,
            )
            for r in retrievers_in
        ]
        for s in shots
    }
    tbl = pd.DataFrame(str_data, index=pd.Index(retrievers_in, name="Retriever"))

    def _color_col(col):
        return [
            f"background-color: {color_fn(num_tbl.at[r, col.name] if col.name in num_tbl.columns else np.nan)}"
            for r in retrievers_in
        ]

    return (
        tbl.style
        .apply(_color_col, axis=0)
        .set_properties(**{"text-align": "center", "color": "#111",
                           "font-size": "13px", "min-width": "110px"})
        .set_table_styles(TABLE_STYLES)
        .set_caption(title)
    )

In [7]:
models_present = [m for m in MODELS_ORDER if m in mean_acc["model"].values]

for model in models_present:
    sub = mean_acc[mean_acc["model"] == model]
    display(build_model_table(
        sub, RETRIEVER_ORDER,
        title=f"{model} — Accuracy (%) by retriever & number of shots",
    ))

,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,49.5%,,,,,
random,,46.8%,48.0%,48.5%,48.1%,47.2%
text,,50.0%,48.1%,49.3%,47.1%,47.1%
vision_ts,,50.7%,50.5%,51.3%,50.5%,50.7%
ts,,49.3%,51.6%,51.7%,49.7%,48.7%
rrf,,47.2%,48.3%,47.9%,48.4%,46.9%
delay_dino,,50.0%,50.7%,52.0%,52.0%,49.7%
rrf-text-delay_dino,,47.7%,49.9%,49.3%,47.2%,47.2%
rrf-ts-delay_dino,,50.0%,51.7%,51.9%,50.1%,49.5%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,58.8%,,,,,
random,,61.7%,61.8%,60.2%,59.4%,61.0%
text,,64.6%,63.8%,64.3%,64.3%,65.7%
vision_ts,,63.8%,63.0%,64.2%,65.1%,65.7%
ts,,61.9%,63.9%,63.4%,64.1%,64.5%
rrf,,59.8%,61.5%,61.7%,62.2%,61.1%
delay_dino,,61.8%,64.6%,65.0%,65.3%,64.6%
rrf-text-delay_dino,,59.5%,60.2%,63.1%,61.9%,61.0%
rrf-ts-delay_dino,,61.0%,64.5%,63.3%,63.8%,63.1%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,48.0% ±0.1,,,,,
random,,47.6% ±0.1,48.3%,49.6%,50.5%,51.1%
text,,49.7% ±0.2,50.8% ±0.2,51.9%,52.1%,51.1%
vision_ts,,51.7%,53.8%,52.9%,54.4%,56.2%
ts,,50.0% ±0.3,53.4% ±0.1,54.0%,55.4%,54.7%
rrf,,48.7% ±0.2,52.1%,52.5%,51.5%,52.0%
delay_dino,,48.9% ±0.2,53.6%,54.0%,54.3%,55.4%
rrf-text-delay_dino,,49.9%,51.0% ±0.2,51.1%,52.4%,52.9%
rrf-ts-delay_dino,,51.6% ±0.1,54.0% ±0.1,53.6%,53.2%,55.9%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,39.1% ±3.6,,,,,
random,,39.1% ±3.6,39.1% ±3.6,39.1% ±3.6,39.1% ±3.6,39.1% ±3.6
text,,39.1% ±3.6,39.1% ±3.6,37.5% ±3.3,39.1% ±3.6,39.1% ±3.6
vision_ts,,42.2%,39.1% ±3.6,37.5% ±3.3,39.1% ±3.6,39.1% ±3.6
ts,,39.1% ±3.6,38.7% ±5.0,38.7% ±5.0,39.1% ±3.6,39.1% ±3.6


---
## 4  Gain over 0-shot baseline (Δ) — one table per model

Rows: retriever (excl. `none`) · Columns: shots · Values: change vs 0-shot  
Green = helps · Red = hurts

In [8]:
kshot_retrievers = [r for r in RETRIEVER_ORDER if r != "none"]
delta_df = mean_acc[mean_acc["retriever"] != "none"].copy()

for model in models_present:
    sub = delta_df[delta_df["model"] == model]
    display(build_model_table(
        sub, kshot_retrievers,
        title=f"{model} — Δ accuracy vs 0-shot baseline (positive = retrieval helps)",
        value_col="delta", std_col="acc_std",
        fmt_fn=_fmt_delta, color_fn=_delta_color,
    ))

,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,
random,-2.7pp,-1.5pp,-0.9pp,-1.3pp,-2.3pp
text,+0.5pp,-1.3pp,-0.1pp,-2.4pp,-2.4pp
vision_ts,+1.2pp,+1.1pp,+1.9pp,+1.1pp,+1.2pp
ts,-0.1pp,+2.1pp,+2.3pp,+0.3pp,-0.8pp
rrf,-2.3pp,-1.2pp,-1.6pp,-1.1pp,-2.5pp
delay_dino,+0.5pp,+1.2pp,+2.5pp,+2.5pp,+0.3pp
rrf-text-delay_dino,-1.7pp,+0.4pp,-0.1pp,-2.3pp,-2.3pp
rrf-ts-delay_dino,+0.5pp,+2.3pp,+2.4pp,+0.7pp,+0.0pp
rrf-text-ts-vision_ts-spectral-stats-vision_wavelet,-1.7pp,-0.5pp,+0.5pp,+0.7pp,+0.3pp


,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,
random,+2.8pp,+2.9pp,+1.3pp,+0.5pp,+2.1pp
text,+5.8pp,+5.0pp,+5.5pp,+5.5pp,+6.8pp
vision_ts,+5.0pp,+4.2pp,+5.4pp,+6.3pp,+6.8pp
ts,+3.1pp,+5.1pp,+4.6pp,+5.2pp,+5.6pp
rrf,+0.9pp,+2.7pp,+2.8pp,+3.4pp,+2.3pp
delay_dino,+2.9pp,+5.8pp,+6.2pp,+6.4pp,+5.8pp
rrf-text-delay_dino,+0.7pp,+1.3pp,+4.3pp,+3.1pp,+2.1pp
rrf-ts-delay_dino,+2.1pp,+5.6pp,+4.4pp,+5.0pp,+4.3pp
rrf-text-ts-vision_ts-spectral-stats-vision_wavelet,+1.6pp,+3.2pp,+4.0pp,+4.0pp,+4.6pp


,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,
random,-0.4pp ±0.1,+0.3pp,+1.6pp,+2.5pp,+3.1pp
text,+1.7pp ±0.2,+2.8pp ±0.2,+3.9pp,+4.2pp,+3.1pp
vision_ts,+3.7pp,+5.8pp,+5.0pp,+6.4pp,+8.2pp
ts,+2.0pp ±0.3,+5.4pp ±0.1,+6.0pp,+7.4pp,+6.7pp
rrf,+0.8pp ±0.2,+4.1pp,+4.6pp,+3.5pp,+4.0pp
delay_dino,+0.9pp ±0.2,+5.6pp,+6.0pp,+6.3pp,+7.4pp
rrf-text-delay_dino,+1.9pp,+3.0pp ±0.2,+3.1pp,+4.4pp,+5.0pp
rrf-ts-delay_dino,+3.6pp ±0.1,+6.0pp ±0.1,+5.6pp,+5.2pp,+7.9pp
rrf-text-ts-vision_ts-spectral-stats-vision_wavelet,+1.6pp ±0.1,+2.4pp,+4.6pp,+2.9pp,+6.0pp


,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,
random,+0.0pp ±3.6,+0.0pp ±3.6,+0.0pp ±3.6,+0.0pp ±3.6,+0.0pp ±3.6
text,+0.0pp ±3.6,+0.0pp ±3.6,-1.6pp ±3.3,+0.0pp ±3.6,+0.0pp ±3.6
vision_ts,+3.2pp,+0.0pp ±3.6,-1.6pp ±3.3,+0.0pp ±3.6,+0.0pp ±3.6
ts,+0.0pp ±3.6,-0.4pp ±5.0,-0.4pp ±5.0,+0.0pp ±3.6,+0.0pp ±3.6


---
## 5  Accuracy by category — one table per model per category

In [9]:
cat_run_acc = (
    df.groupby(["model", "retriever", "num_shots", "seed", "category"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)
cat_mean_acc = (
    cat_run_acc
    .groupby(["model", "retriever", "num_shots", "category"])["accuracy"]
    .agg(acc_mean="mean", acc_std="std")
    .reset_index()
)
cat_mean_acc["acc_std"] = cat_mean_acc["acc_std"].fillna(0)

for category in sorted(df["category"].unique()):
    n_q = df[df["category"] == category]["id"].nunique()
    print(f"\n{'━'*60}")
    print(f"  {category}   ({n_q} questions)")
    print(f"{'━'*60}")
    for model in models_present:
        sub = cat_mean_acc[
            (cat_mean_acc["category"] == category) &
            (cat_mean_acc["model"] == model)
        ].drop(columns="category")
        if sub.empty:
            continue
        display(build_model_table(
            sub, RETRIEVER_ORDER,
            title=f"{model} — {category} accuracy (%)",
        ))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Anolmaly Detection   (108 questions)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,39.8%,,,,,
random,,39.8%,38.0%,45.4%,41.7%,40.7%
text,,46.3%,40.7%,43.5%,41.7%,44.4%
vision_ts,,45.4%,40.7%,40.7%,43.5%,46.3%
ts,,49.1%,49.1%,45.4%,41.7%,42.6%
rrf,,46.3%,43.5%,43.5%,43.5%,50.0%
delay_dino,,49.1%,45.4%,45.4%,44.4%,38.9%
rrf-text-delay_dino,,49.1%,48.1%,45.4%,37.0%,40.7%
rrf-ts-delay_dino,,48.1%,52.8%,47.2%,41.7%,40.7%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,49.1%,,,,,
random,,50.9%,52.8%,51.9%,53.7%,54.6%
text,,51.9%,53.7%,54.6%,57.4%,58.3%
vision_ts,,51.9%,50.0%,53.7%,52.8%,52.8%
ts,,53.7%,56.5%,54.6%,53.7%,51.9%
rrf,,47.2%,49.1%,50.9%,50.9%,56.5%
delay_dino,,48.1%,55.6%,57.4%,57.4%,53.7%
rrf-text-delay_dino,,47.2%,50.0%,51.9%,47.2%,50.0%
rrf-ts-delay_dino,,46.3%,52.8%,56.5%,52.8%,52.8%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,37.3% ±0.5,,,,,
random,,39.8%,39.8%,41.7%,38.9%,41.7%
text,,42.3% ±0.5,42.3% ±0.5,41.0% ±0.5,42.6%,37.0%
vision_ts,,41.7%,42.6%,42.6%,45.4%,46.3%
ts,,39.5% ±0.5,43.5%,43.5%,45.4%,44.4%
rrf,,46.0% ±0.5,47.2%,42.6%,44.4%,43.5%
delay_dino,,41.4% ±0.5,47.5% ±0.5,48.1%,47.2%,44.4%
rrf-text-delay_dino,,43.8% ±0.5,44.4%,41.7%,42.6%,40.7%
rrf-ts-delay_dino,,45.4%,47.2%,45.4%,42.6%,46.3%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,32.1% ±4.8,,,,,
random,,32.1% ±4.8,32.1% ±4.8,32.1% ±4.8,32.1% ±4.8,32.1% ±4.8
text,,32.1% ±4.8,32.1% ±4.8,30.1% ±4.6,32.1% ±4.8,32.1% ±4.8
vision_ts,,36.1%,32.1% ±4.8,30.1% ±4.6,32.1% ±4.8,32.1% ±4.8
ts,,32.1% ±4.8,31.5% ±6.5,31.5% ±6.5,32.1% ±4.8,32.1% ±4.8


...


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Similarity Analysis   (120 questions)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,55.8%,,,,,
random,,46.7%,50.0%,52.5%,53.3%,52.5%
text,,55.0%,56.7%,55.8%,51.7%,50.8%
vision_ts,,51.7%,54.2%,59.2%,50.0%,55.0%
ts,,49.2%,55.0%,55.0%,56.7%,53.3%
rrf,,45.8%,55.0%,51.7%,50.8%,49.2%
delay_dino,,49.2%,53.3%,56.7%,55.8%,56.7%
rrf-text-delay_dino,,50.0%,58.3%,55.8%,58.3%,55.8%
rrf-ts-delay_dino,,51.7%,53.3%,55.8%,57.5%,55.8%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,66.7%,,,,,
random,,70.0%,70.8%,65.0%,65.8%,66.7%
text,,70.8%,77.5%,77.5%,71.7%,75.8%
vision_ts,,75.0%,71.7%,75.0%,71.7%,74.2%
ts,,64.2%,69.2%,66.7%,70.8%,72.5%
rrf,,65.8%,72.5%,71.7%,70.0%,66.7%
delay_dino,,68.3%,70.8%,74.2%,71.7%,74.2%
rrf-text-delay_dino,,69.2%,73.3%,73.3%,75.0%,72.5%
rrf-ts-delay_dino,,70.8%,73.3%,73.3%,70.0%,70.0%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,51.7%,,,,,
random,,52.5%,55.0%,57.5%,55.8%,55.0%
text,,55.8%,57.5%,60.0%,62.5%,61.7%
vision_ts,,58.3%,60.8%,60.0%,60.8%,60.8%
ts,,54.2%,61.7%,60.8%,60.8%,60.0%
rrf,,51.7%,56.7%,63.3%,59.2%,58.3%
delay_dino,,55.0%,56.7%,56.7%,60.0%,60.8%
rrf-text-delay_dino,,58.3%,54.2%,61.7%,60.8%,63.3%
rrf-ts-delay_dino,,60.8%,64.2%,60.8%,60.0%,64.2%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,46.1% ±2.5,,,,,
random,,46.1% ±2.5,46.1% ±2.5,46.1% ±2.5,46.1% ±2.5,46.1% ±2.5
text,,46.1% ±2.5,46.1% ±2.5,45.8% ±3.5,46.1% ±2.5,46.1% ±2.5
vision_ts,,46.7%,46.1% ±2.5,45.8% ±3.5,46.1% ±2.5,46.1% ±2.5
ts,,46.1% ±2.5,45.0% ±2.4,45.0% ±2.4,46.1% ±2.5,46.1% ±2.5


---
## 6  Accuracy by difficulty

One table per difficulty level · same layout as §3.

In [10]:
diff_run_acc = (
    df.groupby(["model", "retriever", "num_shots", "seed", "difficulty"])["correct"]
    .mean()
    .reset_index(name="accuracy")
)
diff_mean_acc = (
    diff_run_acc
    .groupby(["model", "retriever", "num_shots", "difficulty"])["accuracy"]
    .agg(acc_mean="mean", acc_std="std")
    .reset_index()
)
diff_mean_acc["acc_std"] = diff_mean_acc["acc_std"].fillna(0)

n_by_diff = df.groupby("difficulty")["id"].nunique()

for diff in DIFF_ORDER:
    n_q = n_by_diff.get(diff, "?")
    print(f"\n{'━'*60}")
    print(f"  {diff.capitalize()}   ({n_q} questions)")
    print(f"{'━'*60}")
    for model in models_present:
        sub = diff_mean_acc[
            (diff_mean_acc["difficulty"] == diff) &
            (diff_mean_acc["model"] == model)
        ].drop(columns="difficulty")
        if sub.empty:
            continue
        display(build_model_table(
            sub, RETRIEVER_ORDER,
            title=f"{model} — Accuracy (%) on {diff} questions",
        ))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Easy   (236 questions)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,52.1%,,,,,
random,,47.9%,52.1%,50.0%,50.0%,48.7%
text,,52.5%,52.5%,53.0%,48.3%,48.7%
vision_ts,,51.3%,53.0%,53.8%,55.5%,54.2%
ts,,49.6%,53.4%,54.2%,53.0%,52.1%
rrf,,50.0%,53.0%,52.1%,50.4%,47.9%
delay_dino,,50.8%,52.5%,55.5%,55.1%,55.5%
rrf-text-delay_dino,,48.7%,52.1%,50.8%,47.9%,49.6%
rrf-ts-delay_dino,,51.3%,51.3%,53.4%,53.0%,50.4%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,62.7%,,,,,
random,,64.4%,63.1%,61.9%,58.5%,61.9%
text,,66.9%,65.7%,64.0%,61.4%,64.0%
vision_ts,,69.9%,68.2%,66.5%,68.6%,69.1%
ts,,65.3%,66.9%,64.8%,65.7%,66.5%
rrf,,64.0%,66.1%,64.4%,63.6%,60.2%
delay_dino,,65.3%,68.2%,68.2%,67.4%,68.2%
rrf-text-delay_dino,,61.0%,62.7%,66.1%,64.0%,61.0%
rrf-ts-delay_dino,,65.7%,69.5%,64.0%,65.7%,64.8%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,55.4% ±0.2,,,,,
random,,53.8%,53.4%,56.4%,55.5%,56.8%
text,,52.4% ±0.6,58.3% ±0.5,61.9%,61.9%,58.5%
vision_ts,,62.1% ±0.2,64.8%,63.6%,63.1%,64.8%
ts,,55.9% ±0.7,59.2% ±0.2,60.2%,63.6%,63.6%
rrf,,55.1% ±0.4,58.6% ±0.2,58.5%,56.8%,58.5%
delay_dino,,54.0% ±0.2,55.9%,60.2%,60.6%,64.8%
rrf-text-delay_dino,,53.5% ±0.2,54.7% ±0.4,53.8%,57.6%,56.8%
rrf-ts-delay_dino,,56.9% ±0.2,60.6%,60.2%,60.6%,62.7%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,39.4% ±3.5,,,,,
random,,39.4% ±3.5,39.4% ±3.5,39.4% ±3.5,39.4% ±3.5,39.4% ±3.5
text,,39.4% ±3.5,39.4% ±3.5,39.0% ±4.8,39.4% ±3.5,39.4% ±3.5
vision_ts,,40.3%,39.4% ±3.5,39.0% ±4.8,39.4% ±3.5,39.4% ±3.5
ts,,39.4% ±3.5,37.9% ±3.3,37.9% ±3.3,39.4% ±3.5,39.4% ±3.5


...


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Hard   (486 questions)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,48.1%,,,,,
random,,46.3%,45.7%,48.1%,47.1%,46.7%
text,,49.0%,46.1%,47.9%,46.1%,46.3%
vision_ts,,50.2%,49.2%,50.2%,48.4%,49.2%
ts,,49.0%,50.8%,50.6%,48.4%,46.9%
rrf,,46.1%,46.1%,46.3%,47.9%,46.7%
delay_dino,,49.4%,49.8%,50.6%,50.8%,47.3%
rrf-text-delay_dino,,47.5%,48.8%,49.0%,46.7%,46.1%
rrf-ts-delay_dino,,49.6%,52.3%,51.4%,48.8%,48.6%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,57.0%,,,,,
random,,59.9%,60.7%,59.1%,59.3%,60.1%
text,,62.8%,62.3%,63.8%,65.0%,65.8%
vision_ts,,60.7%,60.3%,62.6%,63.0%,63.6%
ts,,59.5%,62.1%,61.9%,62.8%,63.0%
rrf,,56.8%,58.6%,59.7%,61.1%,60.9%
delay_dino,,59.3%,62.8%,63.0%,64.0%,62.1%
rrf-text-delay_dino,,58.0%,58.8%,61.5%,60.5%,60.7%
rrf-ts-delay_dino,,57.6%,61.5%,62.3%,62.3%,62.1%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,44.1% ±0.1,,,,,
random,,44.2% ±0.2,45.5%,46.1%,47.3%,48.4%
text,,47.4% ±0.1,46.9% ±0.4,47.4% ±0.1,47.7%,47.3%
vision_ts,,46.7% ±0.2,48.1%,48.1%,50.2%,51.9%
ts,,47.3% ±0.2,50.1% ±0.1,51.2%,51.2%,50.4%
rrf,,45.8% ±0.1,48.8%,50.2%,49.0%,48.8%
delay_dino,,46.5% ±0.4,52.6% ±0.1,51.2%,51.0%,51.2%
rrf-text-delay_dino,,47.6% ±0.1,48.6% ±0.1,49.4%,50.2%,51.2%
rrf-ts-delay_dino,,48.9% ±0.1,51.0% ±0.2,50.2%,49.6%,52.7%


,0-shot,1-shot,2-shot,3-shot,5-shot,8-shot
Retriever,,,,,,
none,38.8% ±4.4,,,,,
random,,38.8% ±4.4,38.8% ±4.4,38.8% ±4.4,38.8% ±4.4,38.8% ±4.4
text,,38.8% ±4.4,38.8% ±4.4,36.6% ±3.5,38.8% ±4.4,38.8% ±4.4
vision_ts,,43.0%,38.8% ±4.4,36.6% ±3.5,38.8% ±4.4,38.8% ±4.4
ts,,38.8% ±4.4,38.6% ±6.3,38.6% ±6.3,38.8% ±4.4,38.8% ±4.4
